# Phase 1 — Embedding + Qdrant index

Second half of the Phase 1 vertical slice. Input is `data/phase1/` as written by
`ingest_phase1.ipynb`; output is a Qdrant collection that dense retrieval can query.
This notebook never opens a PDF and never calls a VLM — if it needs either, the hand-off
boundary is in the wrong place.

**Decisions made here, with the reasoning that has to survive questioning:**

| Decision | Why |
|---|---|
| BGE-M3 via `sentence-transformers`, local | ROADMAP §1. Free and rate-limit-free, so re-embedding after a chunking change costs time, not money — which is what makes the chunking ablation affordable |
| Dense only, no sparse/ColBERT head | BGE-M3 emits all three, but Phase 1 is deliberately dense-only (ROADMAP Phase 1) so row 1 of the ablation table is a real floor. BM25 arrives in Phase 3 as a separate index |
| No query instruction prefix | BGE-M3 is trained without one, unlike `bge-large-en-v1.5` which needs *"Represent this sentence for searching relevant passages:"*. Adding a prefix here degrades retrieval rather than helping |
| Cosine distance, pre-normalised vectors | Normalised vectors make cosine and dot product identical, so the reranker and any later score-threshold work on the same scale |
| Named vector `dense` from the start | Qdrant fixes its vector schema at collection creation. Naming it now means Phase 3's sparse vector and any image vector are additions, not a schema migration |
| Embeddings cached to `.npy` | CPU embedding is the slow step. Qdrant work gets re-run often; embedding should not be re-run with it |

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "phase1"
CHUNKS_PATH = DATA_DIR / "chunks.jsonl"
DOCS_PATH = DATA_DIR / "documents.jsonl"
EMBEDDINGS_PATH = DATA_DIR / "embeddings.npy"
EMBEDDING_IDS_PATH = DATA_DIR / "embedding_ids.json"

EMBED_MODEL_ID = "BAAI/bge-m3"
COLLECTION = "arxiv_phase1"
# CPU-sized. Raise to 32-64 the moment this runs on a GPU; it is the only knob that matters
# for embedding wall-clock here.
EMBED_BATCH_SIZE = 8
UPSERT_BATCH_SIZE = 256

# --- Qdrant deployment ----------------------------------------------------------------------
# "local" is the embedded client: no server, no Docker, storage is just a directory. Same client
# API as the server, real payload filtering, real named vectors.
#
# What it does NOT do: build an HNSW graph. Local mode searches exhaustively, so `m` /
# `ef_construct` are accepted and ignored. At 500-5,000 chunks exact search is instant and
# strictly higher recall than an approximate index, so nothing is lost at Phase 1 scale - but the
# ROADMAP's "tunable HNSW" claim is a *server* property, and any latency number measured here is
# not the latency number the real system has. Both facts belong in the write-up.
#
# Switching is one env var - no code change, because everything below goes through `client`.
QDRANT_MODE = os.getenv("QDRANT_MODE", "local")
QDRANT_PATH = PROJECT_ROOT / "data" / "qdrant"

print(f"data dir : {DATA_DIR} exists={DATA_DIR.exists()}")
print(f"qdrant   : mode={QDRANT_MODE}")

D:\.tutorials\agentic-rag-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data dir : D:\.tutorials\agentic-rag-capstone\data\phase1 exists=True
qdrant   : mode=local


## Load the ingested chunks

The telemetry printed here is the same telemetry from `PHASE1_NOTES.md` §7, re-derived from
disk. It runs on every load on purpose: the moment the on-disk distribution stops matching what
ingestion reported, something upstream changed silently.

In [2]:
chunks = [json.loads(line) for line in CHUNKS_PATH.read_text(encoding="utf-8").splitlines()]
#Metadata for each document, keyed by arxiv_id. Each document has a list of images and tables.
documents = {d["arxiv_id"]: d for d in
             (json.loads(line) for line in DOCS_PATH.read_text(encoding="utf-8").splitlines())}

texts = [c["text"] for c in chunks]
ids = [c["id"] for c in chunks]
assert len(set(ids)) == len(ids), "chunk IDs are not unique - upserting would silently drop chunks"

def dist(key):
    counts = {}
    for c in chunks:
        counts[c[key]] = counts.get(c[key], 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))

print(f"{len(documents)} documents -> {len(chunks)} chunks")
print("modality      :", dist("modality"))
print("category      :", dist("category"))
print("section       :", dist("section"))
print("section_source:", dist("section_source"))

# Figures extracted vs actually indexed. build_chunks() drops NOT_A_FIGURE and caption failures,
# so "we captioned 72 figures" and "72 figures are searchable" are different numbers, and only
# the second one is retrieval coverage.
imgs = [i for d in documents.values() for i in d["images"]]
print(f"\nfigures       : {len(imgs)} extracted, "
      f"{sum(i['indexed'] for i in imgs)} indexed, "
      f"{sum(i['caption'] == 'NOT_A_FIGURE' for i in imgs)} NOT_A_FIGURE, "
      f"{sum(i['caption'] is None for i in imgs)} caption failed")
print(f"tables        : {sum(len(d['tables']) for d in documents.values())} rendered")

20 documents -> 544 chunks
modality      : {'text': 521, 'figure': 16, 'table': 7}
category      : {'ai': 544}
section       : {'conclusion': 386, 'abstract_intro': 158}
section_source: {'toc': 512, 'headings': 32}

figures       : 72 extracted, 16 indexed, 56 NOT_A_FIGURE, 0 caption failed
tables        : 7 rendered


In [3]:
print(documents)

{'2607.28488v1': {'arxiv_id': '2607.28488v1', 'category': 'ai', 'topic': 'SCOPE: Supply-Chain Operations through Coupled Policies for End-to-End Coordination', 'topic_source': 'layout', 'num_pages': 9, 'authors': {'1': 'Yunhao Liang', '2': 'Xianqi Cao', '3': 'Pujun Zhang', '4': 'Yuan Qu', '5': 'Yongzhi Qi', '6': 'Ningxuan Kang', '7': 'Max Z. J. Shen'}, 'section_source': 'headings', 'sections_found': True, 'n_chunks': 13, 'by_modality': {'text': 13}, 'images': [], 'tables': []}, '2607.28503v1': {'arxiv_id': '2607.28503v1', 'category': 'ai', 'topic': 'Bench InfoOps', 'topic_source': 'layout', 'num_pages': 16, 'authors': {'1': 'Dorian Quelle', '2': 'Lisa-Maria Neudert', '3': 'Jonathan Bright', '4': 'John Gallacher'}, 'section_source': 'toc', 'sections_found': True, 'n_chunks': 9, 'by_modality': {'text': 9}, 'images': [], 'tables': []}, '2607.28498v1': {'arxiv_id': '2607.28498v1', 'category': 'ai', 'topic': 'TCA-SIR: Learning Target-Conditioned Abstractions for Scientific Inspiration Retri

## Embedding model

First run downloads ~2.2 GB of weights and caches them under `~/.cache/huggingface`.

In [4]:
t0 = time.perf_counter()
embedder = SentenceTransformer(EMBED_MODEL_ID)
EMBED_DIM = embedder.get_embedding_dimension()

print(f"{EMBED_MODEL_ID} loaded in {time.perf_counter() - t0:.1f}s")
print(f"  device         : {embedder.device}")
print(f"  dimension      : {EMBED_DIM}")
print(f"  max_seq_length : {embedder.max_seq_length} tokens")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32560.46it/s]

BAAI/bge-m3 loaded in 7.8s
  device         : cpu
  dimension      : 1024
  max_seq_length : 8192 tokens


## Truncation audit — before embedding, not after

An embedder silently truncates anything past its token limit. That is the exact failure shape
`PHASE1_NOTES.md` §9 is about: the run succeeds, the vector looks fine, and the tail of a table
is simply not searchable.

Figures and tables bypass the splitter by design (§5), so they are the only chunks that can
exceed `CHUNK_SIZE`. This measures the real risk rather than assuming it away.

In [5]:
token_lengths = np.array([len(t) for t in embedder.tokenizer(texts)["input_ids"]])
limit = embedder.max_seq_length
truncated = np.flatnonzero(token_lengths > limit)

print(f"tokens: min={token_lengths.min()} median={int(np.median(token_lengths))} "
      f"p95={int(np.percentile(token_lengths, 95))} max={token_lengths.max()} (limit {limit})")
print(f"chunks over the limit: {len(truncated)}")
for i in truncated[:10]:
    c = chunks[i]
    print(f"  TRUNCATED {c['id'][:8]} {c['arxiv_id']} {c['modality']:6s} "
          f"{token_lengths[i]} tokens, {token_lengths[i] - limit} lost")

# BGE-M3's 8,192-token window is why this is empty. On a 512-token embedder the long table
# chunks would truncate silently - worth stating as a reason for the model choice, not just
# "it was on the roadmap".
if len(truncated) == 0:
    print("  none - no chunk loses text at embedding time")

tokens: min=33 median=247 p95=346 max=790 (limit 8192)
chunks over the limit: 0
  none - no chunk loses text at embedding time


In [6]:
print(limit)

8192


## Embed

Cached to `embeddings.npy`, keyed by chunk ID order. Re-running this notebook to fix something
in Qdrant should not cost another CPU embedding pass; changing the chunks *should* invalidate
the cache, which is what the ID check does.

In [7]:
def load_cached_embeddings(expected_ids: list[str]) -> np.ndarray | None:
    if not (EMBEDDINGS_PATH.exists() and EMBEDDING_IDS_PATH.exists()):
        return None
    cached_ids = json.loads(EMBEDDING_IDS_PATH.read_text(encoding="utf-8"))
    if cached_ids != expected_ids:
        print(f"cache stale ({len(cached_ids)} ids cached vs {len(expected_ids)} now) - re-embedding")
        return None
    vectors = np.load(EMBEDDINGS_PATH)
    print(f"loaded {vectors.shape} from cache")
    return vectors


vectors = load_cached_embeddings(ids)

if vectors is None:
    t0 = time.perf_counter()
    vectors = embedder.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        # Normalise here, once. Cosine on unit vectors is a dot product, so scores stay
        # comparable across every downstream stage instead of being renormalised ad hoc.
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    elapsed = time.perf_counter() - t0
    np.save(EMBEDDINGS_PATH, vectors)
    EMBEDDING_IDS_PATH.write_text(json.dumps(ids), encoding="utf-8")
    print(f"embedded {len(texts)} chunks in {elapsed:.1f}s "
          f"({len(texts) / elapsed:.1f} chunks/s on {embedder.device})")

assert vectors.shape == (len(chunks), EMBED_DIM), f"unexpected shape {vectors.shape}"
norms = np.linalg.norm(vectors, axis=1)
assert np.allclose(norms, 1.0, atol=1e-3), f"vectors not unit-norm: {norms.min()}..{norms.max()}"
print(f"{vectors.shape} float32, unit-norm confirmed")

loaded (544, 1024) from cache
(544, 1024) float32, unit-norm confirmed


## Sparse embed (BM25)

Phase 3 lands BM25 as a second named vector, `bm25`, computed by fastembed's `Qdrant/bm25` model
— chosen over a separate in-process BM25 index because Qdrant's `Modifier.IDF` maintains corpus
IDF itself as points are added, so ingesting more categories later needs no manual BM25 refit,
and there is still exactly one store behind retrieval. Same caching pattern as the dense
vectors: cached to disk, invalidated by the same chunk-ID list.

In [ ]:
from fastembed import SparseTextEmbedding

SPARSE_MODEL_ID = "Qdrant/bm25"
SPARSE_VECTOR_NAME = "bm25"
SPARSE_INDICES_PATH = DATA_DIR / "bm25_indices.json"
SPARSE_VALUES_PATH = DATA_DIR / "bm25_values.json"

# Captured before the dense-embed cell above can write EMBEDDING_IDS_PATH. Bug found 2026-08-19
# expanding to cs.CL/finance: that cell updates EMBEDDING_IDS_PATH the moment it notices the
# dense cache is stale, *before* this cell runs - so a naive "compare against EMBEDDING_IDS_PATH"
# staleness check here was comparing the new id list against itself and calling a stale 544-length
# BM25 cache "fresh" (PHASE1_NOTES.md §10.1/§10.6 territory: a cache invalidation bug, not a
# chunking-strategy one). Snapshotting the on-disk value first, before dense embedding mutates it,
# is what makes this check trustworthy again.
_ids_on_disk_before_dense_embed = (
    json.loads(EMBEDDING_IDS_PATH.read_text(encoding="utf-8")) if EMBEDDING_IDS_PATH.exists() else None
)


def load_cached_sparse(expected_ids: list[str]):
    if not (SPARSE_INDICES_PATH.exists() and SPARSE_VALUES_PATH.exists()):
        return None
    if _ids_on_disk_before_dense_embed is None or _ids_on_disk_before_dense_embed != expected_ids:
        print(f"sparse cache stale ({len(_ids_on_disk_before_dense_embed or [])} ids vs "
              f"{len(expected_ids)} now) - re-embedding")
        return None
    indices = json.loads(SPARSE_INDICES_PATH.read_text(encoding="utf-8"))
    values = json.loads(SPARSE_VALUES_PATH.read_text(encoding="utf-8"))
    if len(indices) != len(expected_ids):
        print(f"sparse cache length mismatch ({len(indices)} vs {len(expected_ids)}) - re-embedding")
        return None
    print(f"loaded {len(indices)} sparse vectors from cache")
    return indices, values


t0 = time.perf_counter()
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL_ID)
print(f"{SPARSE_MODEL_ID} loaded in {time.perf_counter() - t0:.1f}s")

cached = load_cached_sparse(ids)
if cached is None:
    t0 = time.perf_counter()
    # Document-side embedding: raw term-frequency weights. IDF is applied server-side by the
    # collection's Modifier.IDF, not baked in here - that's what lets it stay correct as more
    # documents are added later without recomputing every existing sparse vector.
    sparse_vectors = list(sparse_embedder.embed(texts))
    sparse_indices = [v.indices.tolist() for v in sparse_vectors]
    sparse_values = [v.values.tolist() for v in sparse_vectors]
    SPARSE_INDICES_PATH.write_text(json.dumps(sparse_indices), encoding="utf-8")
    SPARSE_VALUES_PATH.write_text(json.dumps(sparse_values), encoding="utf-8")
    print(f"sparse-embedded {len(texts)} chunks in {time.perf_counter() - t0:.1f}s")
else:
    sparse_indices, sparse_values = cached

assert len(sparse_indices) == len(chunks) == len(sparse_values), "sparse vector count mismatch"
nnz = [len(v) for v in sparse_indices]
print(f"{len(sparse_indices)} BM25 sparse vectors ready, "
      f"nnz: min={min(nnz)} median={int(np.median(nnz))} max={max(nnz)}")

In [9]:
# Re-running this cell in a live kernel would otherwise deadlock on the storage lock.
if "client" in dir() and isinstance(globals().get("client"), QdrantClient):
    client.close()

if QDRANT_MODE == "local":
    QDRANT_PATH.mkdir(parents=True, exist_ok=True)
    client = QdrantClient(path=str(QDRANT_PATH))
else:
    client = QdrantClient(
        url=os.getenv("QDRANT_URL", "http://localhost:6333"),
        api_key=os.getenv("QDRANT_API_KEY"),
    )

print(f"connected ({QDRANT_MODE}); existing collections:",
      [c.name for c in client.get_collections().collections])

connected (local); existing collections: ['arxiv_phase1']


In [10]:
# Named vector from the start. Qdrant fixes the vector schema at creation, so an unnamed default
# vector today means dropping and rebuilding the collection when Phase 3 adds a sparse vector for
# hybrid search. Naming it "dense" made that Phase 3 addition below an addition, not a migration.
#
# hnsw_config is set even though embedded mode ignores it (exact search, no graph) - it documents
# the server-mode intent, and flipping QDRANT_MODE then uses these values rather than defaults.
VECTORS_CONFIG = {
    "dense": models.VectorParams(
        size=EMBED_DIM,
        distance=models.Distance.COSINE,
        hnsw_config=models.HnswConfigDiff(m=16, ef_construct=100),
    )
}

# BM25 as a native Qdrant sparse vector rather than a second in-process index: Modifier.IDF has
# Qdrant maintain corpus IDF itself as points are added, so later ingestion needs no manual BM25
# refit, and the fastembed document-side vectors (raw term-frequency weights) stay valid inputs.
SPARSE_VECTORS_CONFIG = {
    SPARSE_VECTOR_NAME: models.SparseVectorParams(modifier=models.Modifier.IDF),
}

if client.collection_exists(COLLECTION):
    existing = client.get_collection(COLLECTION)
    has_bm25 = bool(existing.config.params.sparse_vectors) and (
        SPARSE_VECTOR_NAME in existing.config.params.sparse_vectors
    )
    if has_bm25:
        print(f"'{COLLECTION}' exists ({client.count(COLLECTION).count} points), already has "
              f"'{SPARSE_VECTOR_NAME}' - reusing it. Deterministic chunk IDs make the upsert "
              "below idempotent.")
    else:
        # Qdrant fixes the vector schema at creation - there is no supported way to add a named
        # vector to an existing collection. Recreating is cheap and safe here: chunk IDs are
        # deterministic and the source data still lives in chunks.jsonl, so nothing is lost.
        print(f"'{COLLECTION}' exists but has no '{SPARSE_VECTOR_NAME}' vector - recreating with "
              "both dense and sparse configured (destructive, but fully recoverable from "
              "chunks.jsonl plus the cached embeddings above).")
        client.delete_collection(COLLECTION)
        client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VECTORS_CONFIG,
            sparse_vectors_config=SPARSE_VECTORS_CONFIG,
        )
        print(f"recreated '{COLLECTION}' dim={EMBED_DIM} distance=COSINE "
              f"vectors=['dense', '{SPARSE_VECTOR_NAME}']")
else:
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VECTORS_CONFIG,
        sparse_vectors_config=SPARSE_VECTORS_CONFIG,
    )
    print(f"created '{COLLECTION}' dim={EMBED_DIM} distance=COSINE "
          f"vectors=['dense', '{SPARSE_VECTOR_NAME}']")

'arxiv_phase1' exists (544 points), already has 'bm25' - reusing it. Deterministic chunk IDs make the upsert below idempotent.


In [11]:
# Payload indexes. Not an optimisation - Phase 3 filters by category to build the subagent corpus
# partitions, and Phase 10 slices every metric by category and by modality. Filtering still works
# without them; what they buy is that it keeps working at 10K documents instead of 20.
INDEXED_FIELDS = {
    "arxiv_id": models.PayloadSchemaType.KEYWORD,       # per-document lookup, dedup
    "category": models.PayloadSchemaType.KEYWORD,       # subagent partitions, router ground truth
    "modality": models.PayloadSchemaType.KEYWORD,       # text vs figure vs table slices
    "section": models.PayloadSchemaType.KEYWORD,        # section-aware chunking arm
    "section_source": models.PayloadSchemaType.KEYWORD, # does fallback parsing retrieve worse?
}

# Embedded Qdrant ACCEPTS create_payload_index and then does nothing with it - it warns rather
# than raising. Left alone that prints a tidy "indexed category" for an index that does not
# exist, which is the same shape of lie as a caption pipeline reporting success with zero
# captions. The warning is promoted to output so the declaration and the reality stay distinct.
import warnings

for field, schema in INDEXED_FIELDS.items():
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            client.create_payload_index(collection_name=COLLECTION, field_name=field, field_schema=schema)
        note = f"  (no-op: {caught[0].message})" if caught else ""
        print(f"  {field}{note}")
    except Exception as exc:
        # Reported, never swallowed - a missing index is a silent latency cliff at scale.
        print(f"  {field} FAILED: {type(exc).__name__}: {exc}")

if QDRANT_MODE == "local":
    print("\nDeclared, not built. Embedded mode scans payloads exhaustively; the definitions above "
          "apply the moment QDRANT_MODE=server.")

  arxiv_id  (no-op: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.)
  category
  modality
  section
  section_source

Declared, not built. Embedded mode scans payloads exhaustively; the definitions above apply the moment QDRANT_MODE=server.


## Upsert

Point IDs are the UUIDv5s minted at ingestion, so this is an upsert in the true sense: re-running
it overwrites each chunk with itself instead of doubling the corpus.

In [12]:
PAYLOAD_KEYS = ("arxiv_id", "category", "topic", "section", "modality", "page",
                "figure_ref", "section_source", "ordinal")

t0 = time.perf_counter()
for start in range(0, len(chunks), UPSERT_BATCH_SIZE):
    window = chunks[start:start + UPSERT_BATCH_SIZE]
    client.upsert(
        collection_name=COLLECTION,
        points=[
            models.PointStruct(
                id=c["id"],
                vector={
                    "dense": vectors[start + offset].tolist(),
                    SPARSE_VECTOR_NAME: models.SparseVector(
                        indices=sparse_indices[start + offset],
                        values=sparse_values[start + offset],
                    ),
                },
                # The chunk text lives in the payload: retrieval has to return the text it
                # matched, and the generator cites it. Qdrant is the only store here - there is
                # no second lookup by ID into some other database.
                payload={"text": c["text"], **{k: c[k] for k in PAYLOAD_KEYS}},
            )
            for offset, c in enumerate(window)
        ],
        wait=True,
    )
print(f"upserted {len(chunks)} points in {time.perf_counter() - t0:.1f}s")
print("collection count:", client.count(COLLECTION).count)

upserted 544 points in 91.0s
collection count: 544


In [13]:
# Round-trip check. An upsert that reports success and stores the wrong vector is invisible until
# retrieval quality is mysteriously bad, so verify against the source arrays directly.
assert client.count(COLLECTION).count == len(chunks), "point count does not match chunk count"

probe_idx = [0, len(chunks) // 2, len(chunks) - 1]
stored = client.retrieve(COLLECTION, ids=[ids[i] for i in probe_idx], with_vectors=True, with_payload=True)
by_id = {str(p.id): p for p in stored}
for i in probe_idx:
    point = by_id[ids[i]]
    assert point.payload["text"] == chunks[i]["text"], f"payload text mismatch at {i}"
    assert np.allclose(point.vector["dense"], vectors[i], atol=1e-5), f"dense vector mismatch at {i}"
    # Qdrant returns sparse vectors sorted by index; fastembed's own output order is not
    # guaranteed sorted, so compare as index->value maps rather than raw positional lists.
    stored_sparse = point.vector[SPARSE_VECTOR_NAME]
    stored_map = dict(zip(stored_sparse.indices, stored_sparse.values))
    source_map = dict(zip(sparse_indices[i], sparse_values[i]))
    assert stored_map.keys() == source_map.keys(), f"bm25 indices mismatch at {i}"
    assert all(abs(stored_map[k] - source_map[k]) < 1e-5 for k in source_map), \
        f"bm25 values mismatch at {i}"
print(f"verified {len(probe_idx)} points round-trip (payload text + dense vector + bm25 vector)")

hits = client.count(COLLECTION, count_filter=models.Filter(must=[
    models.FieldCondition(key="modality", match=models.MatchValue(value="figure"))
])).count
print(f"filter check: modality=figure -> {hits} points")

verified 3 points round-trip (payload text + dense vector + bm25 vector)
filter check: modality=figure -> 16 points


## Dense retrieval — the Phase 1 slice, end to end

This is ablation row 1: dense only, no BM25, no reranker. Everything in Phase 3 has to beat
these numbers, so they get looked at by hand before anything is measured automatically.

In [14]:
def embed_query(query: str) -> list[float]:
    """BGE-M3 takes the raw query. No instruction prefix - it is trained without one, and the
    `bge-large-en-v1.5` style prefix ("Represent this sentence for searching...") actively hurts
    here. The asymmetry to watch for is model-specific, not universal."""
    return embedder.encode(query, normalize_embeddings=True).tolist()


def search(query: str, limit: int = 5, category: str | None = None,
           modality: str | None = None) -> list[models.ScoredPoint]:
    conditions = []
    if category:
        conditions.append(models.FieldCondition(key="category", match=models.MatchValue(value=category)))
    if modality:
        conditions.append(models.FieldCondition(key="modality", match=models.MatchValue(value=modality)))
    return client.query_points(
        collection_name=COLLECTION,
        query=embed_query(query),
        using="dense",
        limit=limit,
        query_filter=models.Filter(must=conditions) if conditions else None,
        with_payload=True,
    ).points


def show(query: str, **kwargs) -> None:
    filters = " ".join(f"{k}={v}" for k, v in kwargs.items() if k != "limit" and v)
    print(f"\nQ: {query}" + (f"   [{filters}]" if filters else ""))
    results = search(query, **kwargs)
    if not results:
        print("  (no results)")
    for rank, point in enumerate(results, 1):
        p = point.payload
        ref = f" {p['figure_ref']}" if p["figure_ref"] else ""
        snippet = " ".join(p["text"].split())[:150]
        print(f"  {rank}. {point.score:.4f}  {p['arxiv_id']} <{p['category']}> "
              f"{p['modality']}/{p['section']}{ref}")
        print(f"     {snippet}...")

In [15]:
show("How do multi-agent systems adapt their communication topology?")
show("What are the failure modes of computer-use agents at inference time?")
show("belief revision under limited credibility")


Q: How do multi-agent systems adapt their communication topology?


  1. 0.7351  2607.28527v1 <ai> text/abstract_intro
     Abstract Large language model-based multi-agent systems improve complex problem solving through task decomposition, agent specialization, information ...
  2. 0.6996  2607.28527v1 <ai> text/abstract_intro
     ual agents rather than how multiple agents are organized and communicate. Multi-agent frameworks decompose complex tasks across specialized roles and ...
  3. 0.6915  2607.28527v1 <ai> text/abstract_intro
     self-improve online therefore remains underexplored. This raises a central question: Can a multi-agent system improve its communication topology while...
  4. 0.6876  2607.28527v1 <ai> text/conclusion
     Conclusion We presented MANTA, a framework that enables multi-agent systems to revise their collaboration topology during task execution. By combining...
  5. 0.6868  2607.28527v1 <ai> text/abstract_intro
     L1 Prompt APO, APE, DSPy, MIPRO, PromptBreeder, TextGrad, GEPA L0 Output Self-Refine text capability multi-a

  1. 0.6735  2607.28523v1 <ai> text/abstract_intro
     Abstract Belief update concerns changes in an agent’s beliefs induced by changes in the underlying world. Standard Katsuno–Mendelzon update assumes th...
  2. 0.6668  2607.28523v1 <ai> text/conclusion
     sults establish that the principal classes considered herein are related by proper containment. There- fore, selective credibility-limited belief upda...
  3. 0.6618  2607.28523v1 <ai> text/abstract_intro
     among its credible consequences. Third, we isolate an unrestricted-credibility variant that captures selective belief update independently of credibil...
  4. 0.6579  2607.28523v1 <ai> text/conclusion
     9 Conclusion This article introduced selective credibility-limited belief update, a non-prioritized framework that builds on three principal lines of ...
  5. 0.6547  2607.28523v1 <ai> text/abstract_intro
     expressive account of belief update, encompassing established approaches while supporting source-dependent selec

In [16]:
# Filters are what the Phase 5 topic subagents run on, and the modality filter is what makes
# "text-only vs figure-requiring" (§3-6) a measurable slice rather than an assertion.
show("architecture diagram of the proposed pipeline", modality="figure")
show("benchmark results table", modality="table")
show("reinforcement learning for safety", category="ai", limit=3)


Q: architecture diagram of the proposed pipeline   [modality=figure]
  1. 0.4638  2607.28609v1 <ai> figure/conclusion 2607.28609v1_p49_i4
     This figure shows a terminal window displaying the configuration parameters for an agent-based operating system automation framework, including model ...
  2. 0.4544  2607.28609v1 <ai> figure/conclusion 2607.28609v1_p50_i2
     This image displays a mobile interface for a mapping application showing a list of nearby gas stations and their locations on an interactive map. Each...
  3. 0.4400  2607.28526v1 <ai> figure/conclusion 2607.28526v1_p7_i11
     This figure displays an upside-down, street-level photograph of an urban environment featuring multistory buildings on either side of a road. It provi...
  4. 0.4375  2607.28526v1 <ai> figure/conclusion 2607.28526v1_p7_i10
     This image displays an inverted, wide-angle view of a city street lined with multi-story residential and commercial buildings. The perspective capture...
  5. 0.4338  2607.

  1. 0.5626  2607.28580v1 <ai> table/conclusion 2607.28580v1_p7_t0
     | RetrievalPerformance GenerationPerformance hitectureVariant MMQA WebQA Qwen3-VL -4B Qwen3-VL -8B R@2 R@5 R@2 R@5 EM F1 R -L BERTSc EM F1 R -L BERTSc...
  2. 0.5082  2607.28553v1 <ai> table/conclusion 2607.28553v1_p11_t0
     | Optimizer LearningRate BatchSize GroupSize(G) Spectral Reward Weight (λ spec) Entropy Reward Weight (λ phys) KLPenaltyCoefficient(β) TrainingSteps |...
  3. 0.4978  2607.28628v1 <ai> table/conclusion 2607.28628v1_p57_t0
     | | C (SR ) peak peak | | C (SR ) pred pred | | C /C pred train | | | --- | --- | --- | --- | --- | --- | --- | | | EER | EW | EER | EW | EER | EW | |...
  4. 0.4704  2607.28628v1 <ai> table/conclusion 2607.28628v1_p36_t0
     | Dataset | Pathfi nder | SR (%) | ER | EER | | --- | --- | --- | --- | --- | | ID dataset | DGNN | 99 94 . | 430 25 . | 429 99 . | | | AGNN | 76 12 ....
  5. 0.4620  2607.28628v1 <ai> table/conclusion 2607.28628v1_p36_t1
     | Dataset | Pathfi n

## BM25 and hybrid search — the case for Phase 3

Dense embeddings compress a passage into one semantic direction; an exact identifier, symbol,
or acronym is a single token that carries almost no distributional meaning on its own, so cosine
similarity treats it like near-noise. BM25 has the opposite failure mode - it cannot see meaning
at all, only token overlap. Two queries below, with the honest result of each:

- `"2607.28623v1"` (the paper's own arXiv ID) - **neither** method finds the paper. The ID string
  is a page-1 cover stamp, and `PHASE1_NOTES.md` §3.5 explicitly strips it as marginalia during
  ingestion, so it never enters the indexed text at all. This is the useful negative result: an
  identifier that was never retrievable text cannot be retrieved by any retriever, and BM25 does
  not conjure text that isn't there.
- `"PAC-MAN"` (the paper's own acronym, which *does* appear in the body text) - both dense and
  BM25 independently surface the same paper's chunks. Not a case of one beating the other; it is
  a case where a short, low-semantic-content token still round-trips through the dense embedder
  well enough that the two agree. RRF fusion still adds value here by reinforcing chunks both
  branches found, which is visible in the fused ranking below.

In [17]:
def search_bm25(query: str, limit: int = 5, category: str | None = None,
                 modality: str | None = None) -> list[models.ScoredPoint]:
    conditions = []
    if category:
        conditions.append(models.FieldCondition(key="category", match=models.MatchValue(value=category)))
    if modality:
        conditions.append(models.FieldCondition(key="modality", match=models.MatchValue(value=modality)))
    q = next(iter(sparse_embedder.query_embed([query])))
    return client.query_points(
        collection_name=COLLECTION,
        query=models.SparseVector(indices=q.indices.tolist(), values=q.values.tolist()),
        using=SPARSE_VECTOR_NAME,
        limit=limit,
        query_filter=models.Filter(must=conditions) if conditions else None,
        with_payload=True,
    ).points


def reciprocal_rank_fusion(ranked_id_lists: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """Same math as rag/fusion.py - duplicated here rather than imported so this notebook stays a
    self-contained record of the decision, independent of the rag package's later evolution."""
    scores: dict[str, float] = {}
    for ranked in ranked_id_lists:
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: -kv[1])


def show_hybrid(query: str, limit: int = 5, **kwargs) -> None:
    """Dense hits, BM25 hits, and the RRF fusion of both, side by side."""
    dense_hits = search(query, limit=limit, **kwargs)
    bm25_hits = search_bm25(query, limit=limit, **kwargs)
    by_id = {str(p.id): p for p in (*dense_hits, *bm25_hits)}
    fused = reciprocal_rank_fusion(
        [[str(p.id) for p in dense_hits], [str(p.id) for p in bm25_hits]]
    )[:limit]

    def tags(hits):
        return [f"{p.payload['arxiv_id']}" for p in hits] or ["(none)"]

    print(f"\nQ: {query}")
    print("  dense only :", tags(dense_hits))
    print("  bm25  only :", tags(bm25_hits))
    for rank, (cid, score) in enumerate(fused, 1):
        p = by_id[cid].payload
        snippet = " ".join(p["text"].split())[:120]
        print(f"  {rank}. rrf={score:.4f}  {p['arxiv_id']} <{p['category']}> {p['modality']}/{p['section']}")
        print(f"     {snippet}...")

In [18]:
show_hybrid("2607.28623v1")   # exact arXiv ID - a pure token match, no semantic content at all
show_hybrid("PAC-MAN")        # a paper's own acronym - short, near-meaningless to an embedder


Q: 2607.28623v1
  dense only : ['2607.28580v1', '2607.28553v1', '2607.28573v1', '2607.28628v1', '2607.28573v1']
  bm25  only : ['2607.28520v1']
  1. rrf=0.0164  2607.28580v1 <ai> table/conclusion
     | RetrievalPerformance GenerationPerformance hitectureVariant MMQA WebQA Qwen3-VL -4B Qwen3-VL -8B R@2 R@5 R@2 R@5 EM F1...
  2. rrf=0.0164  2607.28520v1 <ai> text/conclusion
     fect information extensive-form games with reinforcement learning. In Proceedings of the 41st International Conference o...
  3. rrf=0.0161  2607.28553v1 <ai> text/conclusion
     Cα bb-w bb Cα-w Cα bb-w bb Cα-w Cα bb-w bb VP Path 2.71 2.00 2.56 2.06 1.11 0.95 1.08 0.96 1.32 0.99 1.39 1.08 OT Path 2...
  4. rrf=0.0159  2607.28573v1 <ai> text/conclusion
     //arxiv.org/abs/2510.00536. Kevin Qinghong Lin, Linjie Li, Difei Gao, Zhengyuan Yang, Shiwei Wu, Zechen Bai, Weixian Lei...
  5. rrf=0.0156  2607.28628v1 <ai> text/conclusion
     3 1.00 1.00 1.00 1.00 1.00 1.00 1.00 1.00 1.00 1.00 1.00 1.00 4 0.8 1.00 1.00 

## What this notebook does not yet do

Carried into the next step rather than quietly skipped:

1. **No generation.** Phase 1 finishes with retrieved chunks → one generation call → a cited
   answer, then the CLI. This notebook stops at retrieved chunks.
2. **Retrieval quality is eyeballed, not measured.** Every number above is a cosine score with
   no ground truth behind it. That is Phase 4's job, and ROADMAP §8 is explicit that eval ships
   before the agent layer.
3. **`ai/`, `cs.CL/`, `finance/` indexed (2026-08-19) — 54 unique docs, 1,219 chunks, 20/category,
   not Phase 2's ~2,000/category "at scale" target.** `astronomy`/`hep-th` have never been through
   the parser (`PHASE1_NOTES.md` §Phase 1 note). Re-running the ingest batch plus this notebook
   upserts cleanly over existing points **only when a document's chunk count doesn't change
   between runs** — see `PHASE1_NOTES.md` §10.1 (ordinal drift from ordinary VLM
   non-determinism, not just a deliberate re-chunk) and §10.6 (`delete_collection()` does not
   purge embedded-mode on-disk storage — delete `data/qdrant/` itself for a guaranteed-clean
   rebuild) before assuming a re-run is free.
4. **Exact search, not HNSW; payload indexes declared, not built.** Embedded mode does neither.
   Recall measured here is therefore a ceiling rather than a realistic figure, and no latency
   number from this notebook describes the deployed system. Flipping `QDRANT_MODE=server`
   activates both — do it before quoting either number.
5. **Figure chunks are captions, not images.** Retrieval matches VLM text; the original bytes sit
   in `data/phase1/images/` and are resolved by `figure_ref` when the generator needs them.